# Arabic-German live translation

**For MOSS, select an A100 (40 GB) or another GPU with at least 32 GiB recommended, then `Runtime > Run all`. The default model combination cannot fit a T4.**

1. Upload `.env` or add its values as Colab Secrets. It must include the three LiveKit credentials and `HF_TOKEN`.
2. The notebook installs Cohere Transcribe Arabic, Qwen3, and MOSS-TTS dependencies. Only the Cohere repository is gated: accept its terms before running.
3. **Speech engine.** `TTS_BACKEND=moss` (default) uses MOSS-TTS: Apache-2.0, built-in German voice, no reference clip needed. `xtts` is the alternative. The engine loads at startup, before you join, so the first spoken turn is fast.

4. When `READY` appears, open the web app on your PC, enable German speech output if desired, and start the microphone.

In [ ]:
%pip install -q python-dotenv


In [ ]:
import importlib
import io, os
from pathlib import Path
from dotenv import dotenv_values, set_key

REQUIRED = ("LIVEKIT_URL", "LIVEKIT_API_KEY", "LIVEKIT_API_SECRET", "HF_TOKEN")
OPTIONAL = (
    "COHERE_MODEL", "COHERE_4BIT", "COHERE_LANGUAGE", "LIVEKIT_ROOM",
    "TRANSLATION_BACKEND", "TRANSLATION_MODEL", "TRANSLATION_4BIT", "TRANSLATION_DTYPE", "TRANSLATION_SOURCE_LANG",
    "TRANSLATION_TARGET_LANG", "TRANSLATION_CONTEXT_TURNS", "TRANSLATION_SYSTEM_PROMPT",
    "TTS_BACKEND", "ASR_DEVICE", "TRANSLATION_DEVICE", "TTS_DEVICE",
    "XTTS_SPEAKER", "XTTS_SPEAKER_WAV",
    "STREAM_TTS", "TTS_FIRST_CHUNK_CHARS", "TTS_CHUNK_CHARS", "TTS_MAX_CHUNK_CHARS",
    "GPU_CONCURRENCY", "INTERIM_CAPTIONS", "AUDIO_QUEUE_MS", "MAX_AUDIO_BACKLOG_SECONDS",
    "MAX_UTTERANCE_SECONDS", "END_SILENCE_SECONDS", "MAX_GEN_TOKENS", "LOG_LEVEL",
    "ASR_TOKENS_PER_SECOND", "TRANSLATION_TOKEN_RATIO", "TTS_MIN_CHUNK_CHARS",
    "MIN_SENTENCE_SECONDS", "SHORT_SEGMENT_SILENCE_SECONDS",
    "MOSS_MODEL", "MOSS_REF_WAV", "MOSS_VQ_DEPTH", "MOSS_TEMPERATURE", "MOSS_TOP_P",
    "MOSS_TOP_K", "MOSS_REPETITION_PENALTY", "MOSS_DTYPE",
)
TARGET = Path("/content/.env")


def parse(text):
    return {k: (v or "").strip() for k, v in dotenv_values(stream=io.StringIO(text), interpolate=False).items() if (v or "").strip()}


def find_existing():
    paths = [TARGET, *sorted(Path("/content").glob("*env*")), *sorted(Path("/content").glob(".env*"))]
    for path in dict.fromkeys(paths):
        if path.is_file():
            values = parse(path.read_text(encoding="utf-8", errors="ignore"))
            if all(values.get(k) for k in REQUIRED):
                return values
    try:
        userdata = importlib.import_module("google.colab.userdata")
    except ImportError:
        print("Colab Secrets are unavailable; checking an uploaded .env instead.")
        return None
    values = {}
    for key in REQUIRED + OPTIONAL:
        try:
            value = (userdata.get(key) or "").strip()
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            if key in REQUIRED:
                print(f"Required Colab Secret {key} is missing or access is disabled.")
            continue
        if value:
            values[key] = value
    if all(values.get(k) for k in REQUIRED):
        return values
    return None


values = find_existing()
if values is None:
    try:
        files = importlib.import_module("google.colab.files")
    except ImportError as exc:
        raise RuntimeError("Run this notebook in Google Colab and upload .env there, or configure Colab Secrets.") from exc
    print("Select a .env containing LIVEKIT_URL, LIVEKIT_API_KEY, LIVEKIT_API_SECRET, and HF_TOKEN.")
    uploaded = files.upload()
    for name, content in uploaded.items():
        parsed = parse(content.decode("utf-8", errors="ignore"))
        if all(parsed.get(k) for k in REQUIRED):
            values = parsed
            break

if values is None:
    raise SystemExit("ERROR: .env must contain the three LIVEKIT_* values and HF_TOKEN.")

TARGET.touch(mode=0o600, exist_ok=True)
os.chmod(TARGET, 0o600)
TARGET.write_text("", encoding="utf-8")
for key, value in values.items():
    set_key(TARGET, key, value, quote_mode="always")
print("OK: credentials found (Host:", values["LIVEKIT_URL"].split("//")[-1], "| models: Cohere + Qwen3)")

In [ ]:
import importlib.metadata
import subprocess, sys
from pathlib import Path
from dotenv import dotenv_values

gpu_check = subprocess.run([sys.executable, "-c", "import torch; assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU (A100 recommended for MOSS)'; print('GPU:', torch.cuda.get_device_name(0))"], capture_output=True, text=True)
print(gpu_check.stdout, end="")
if gpu_check.returncode:
    print(gpu_check.stderr)
    raise RuntimeError("A working CUDA PyTorch runtime is required. Select a GPU runtime and run all cells.")

backend = (dotenv_values("/content/.env", interpolate=False).get("TTS_BACKEND") or "moss").strip().lower()
if backend not in ("moss", "xtts"):
    raise ValueError("TTS_BACKEND must be moss or xtts.")
# Keep Colab's CUDA build; torchaudio must match its torch release.
torch_version = importlib.metadata.version("torch").split("+")[0]
PACKAGES = [f"torch=={torch_version}", f"torchaudio=={torch_version}", "livekit>=1.0", "livekit-api>=1.0", "transformers>=5.4.0,<6", "bitsandbytes>=0.45", "python-dotenv", "numpy>=1.26", "huggingface_hub", "soundfile", "librosa", "sentencepiece", "protobuf", "accelerate", "silero-vad>=5.1", "einops", "tiktoken"]
if backend == "xtts":
    PACKAGES.append("coqui-tts>=0.27")
IMPORT_CHECK = "import livekit.rtc, livekit.api, transformers, bitsandbytes, dotenv, numpy, torch, torchaudio, soundfile, librosa, sentencepiece, silero_vad, einops, tiktoken; from transformers import AutoProcessor, AutoModel, AutoModelForCausalLM, AutoTokenizer, CohereAsrForConditionalGeneration; from transformers.modeling_utils import PreTrainedAudioTokenizerBase"
if backend == "xtts":
    IMPORT_CHECK += "; from TTS.api import TTS"

install_log = Path("/content/install.log")
print(f"Installing dependencies for {backend}; full output: {install_log}", flush=True)
with install_log.open("w", encoding="utf-8") as log:
    result = subprocess.run([sys.executable, "-m", "pip", "install", "--no-input", *PACKAGES], stdout=log, stderr=subprocess.STDOUT)
if result.returncode:
    print(install_log.read_text(encoding="utf-8")[-12000:])
    raise RuntimeError("Installation failed. Full details are in /content/install.log.")
check = subprocess.run([sys.executable, "-c", IMPORT_CHECK], capture_output=True, text=True)
if check.returncode:
    with install_log.open("a", encoding="utf-8") as log:
        log.write(check.stdout + check.stderr)
    print(check.stderr)
    raise RuntimeError("Packages installed but not importable (see above).")
print("OK: Cohere, Qwen3, audio codec, and selected TTS imports verified.")

In [ ]:
%%writefile /content/bot.py
"""Arabic-to-German live speech-to-speech agent for a LiveKit room.

The agent is a streaming pipeline. Every stage is an independent worker with
its own queue, so stage N of one utterance overlaps stage N+1 of the previous
one and the GPU never idles while the CPU works (or vice versa):

    mic track -> VAD thread -> ASR -> translation (token stream)
              -> TTS (clause chunks) -> playout task -> LiveKit audio track

Results are sent back to the speaker as data messages:

  topic "cohere-transcript":  {"type": "interim" | "final", "id": str, "text": str}
  topic "asr-translation":    {"type": "translation", "id": str, "text": str}

The translation is published incrementally while it is generated, and each
finished clause is spoken immediately instead of waiting for the whole
sentence, which is what keeps the spoken output close to real time.

Runs on a single GPU. The default MOSS stack requires a larger GPU than a T4.
"""
from __future__ import annotations

import asyncio
import atexit
import collections
import contextlib
import heapq
import itertools
import json
import logging
import os
import queue
import re
import secrets
import subprocess
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
from dotenv import load_dotenv
from livekit import api, rtc
from silero_vad import load_silero_vad

load_dotenv(Path(__file__).resolve().parent / ".env")

logging.basicConfig(
    level=os.getenv("LOG_LEVEL", "INFO").strip().upper() or "INFO",
    format="%(asctime)s %(levelname)s %(message)s",
    stream=sys.stdout,
)
logger = logging.getLogger("asr-bot")
logging.getLogger("livekit").setLevel(logging.WARNING)

# Fragmentation is the usual reason a T4 runs out of memory halfway through a
# session even though the totals fit; expandable segments avoid that.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


def _env_flag(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in ("1", "true", "yes", "on")


def _env_positive_float(name: str, default: float) -> float:
    """Reads a positive float from the env, falling back to the default."""
    raw = os.getenv(name, "").strip()
    if not raw:
        return default
    try:
        value = float(raw)
    except ValueError:
        logger.warning("Invalid %s=%r; using %s instead", name, raw, default)
        return default
    if value <= 0:
        logger.warning("%s must be > 0; using %s instead", name, default)
        return default
    return value


def _env_int(name: str, default: int, minimum: int = 1) -> int:
    """Reads an integer >= minimum from the env, falling back to the default."""
    raw = os.getenv(name, "").strip()
    if not raw:
        return default
    try:
        value = int(raw)
    except ValueError:
        logger.warning("Invalid %s=%r; using %s instead", name, raw, default)
        return default
    if value < minimum:
        logger.warning("%s must be >= %d; using %s instead", name, minimum, default)
        return default
    return value


ROOM_NAME = os.getenv("LIVEKIT_ROOM", "arabic-asr-live")
BOT_IDENTITY = "arabic-asr-bot"
SETTINGS_TOPIC = "asr-settings"
TRANSLATION_TOPIC = "asr-translation"
TRANSCRIPT_TOPIC = "cohere-transcript"

HF_TOKEN = os.getenv("HF_TOKEN", "").strip() or None
ASR_DEVICE = os.getenv("ASR_DEVICE", "auto").strip().lower()
TRANSLATION_DEVICE = os.getenv("TRANSLATION_DEVICE", "auto").strip().lower()
TTS_DEVICE = os.getenv("TTS_DEVICE", "auto").strip().lower()

# --- ASR -------------------------------------------------------------------
COHERE_MODEL_NAME = os.getenv("COHERE_MODEL", "CohereLabs/cohere-transcribe-arabic-07-2026")
COHERE_4BIT = _env_flag("COHERE_4BIT")
COHERE_LANGUAGE = os.getenv("COHERE_LANGUAGE", "ar").strip() or "ar"

# --- Translation -----------------------------------------------------------
TRANSLATION_MODEL = os.getenv("TRANSLATION_MODEL", "Qwen/Qwen3-4B-Instruct-2507")
TRANSLATION_BACKEND = os.getenv("TRANSLATION_BACKEND", "qwen").strip().lower()
if TRANSLATION_BACKEND not in ("qwen", "translategemma"):
    raise ValueError("TRANSLATION_BACKEND must be qwen or translategemma.")
TRANSLATION_SOURCE_LANG = os.getenv("TRANSLATION_SOURCE_LANG", "ar").strip() or "ar"
TRANSLATION_TARGET_LANG = os.getenv("TRANSLATION_TARGET_LANG", "de").strip() or "de"
# 4-bit NF4 dequantises every weight on every decoded token, which on a T4 is
# 2-4x slower than fp16 *and* costs translation quality. fp16 is therefore the
# default; this flag forces 4-bit, and 4-bit is also used automatically when
# fp16 does not fit into the remaining VRAM.
TRANSLATION_4BIT = _env_flag("TRANSLATION_4BIT")
# Activation/compute dtype of the translator: auto, float16, bfloat16 or float32.
# auto keeps the device default, except for TranslateGemma on GPUs whose default
# is float16 (e.g. T4): Gemma 3 activations exceed float16's 65504 maximum, which
# yields NaN logits and only special tokens, so auto uses bfloat16 there.
TRANSLATION_DTYPE = os.getenv("TRANSLATION_DTYPE", "auto").strip().lower() or "auto"
# System prompt for the translator. The default is tuned for a Friday sermon
# (Jumu'ah khutba); set to an empty value for the plain "translate, no
# explanation" prompt.
TRANSLATION_SYSTEM_PROMPT = os.getenv("TRANSLATION_SYSTEM_PROMPT", "").strip() or (
    "You are a professional simultaneous interpreter rendering a Friday sermon "
    "(Jumu'ah khutba) from {src} to {tgt}. The input arrives in short spoken "
    "fragments and may end mid-sentence: render each one faithfully, keep the "
    "solemn sermon register, keep proper names and Qur'anic phrases in their "
    "standard transliteration, and stay consistent with the earlier turns. "
    "Reply with the {tgt} translation only - no notes, no explanation, no "
    "repetition of the {src} source."
).format(src=TRANSLATION_SOURCE_LANG, tgt=TRANSLATION_TARGET_LANG)
# How many previous (source, translation) pairs to send as chat context so the
# translator stays consistent across turns (0 disables the context). Every turn
# of context is re-prefilled on each request, so this trades latency for
# consistency; 4 is a good balance.
TRANSLATION_CONTEXT_TURNS = _env_int("TRANSLATION_CONTEXT_TURNS", 4, minimum=0)
# Hard ceiling for generated tokens, shared by the ASR and the translator. The
# effective limit per request is derived from the input, so normal turns stop
# long before this; it only exists to stop a degenerate decode.
MAX_GEN_TOKENS = _env_int("MAX_GEN_TOKENS", 448)
# Tokens budgeted per second of audio for the ASR. Speech tops out around five
# words per second and Arabic script costs a few tokens per word, so 25 leaves
# a wide margin over real speech.
ASR_TOKENS_PER_SECOND = _env_int("ASR_TOKENS_PER_SECOND", 25)
# Output-token budget as a multiple of the source length for the translator.
TRANSLATION_TOKEN_RATIO = _env_positive_float("TRANSLATION_TOKEN_RATIO", 3.0)

# --- TTS -------------------------------------------------------------------
TTS_BACKEND = os.getenv("TTS_BACKEND", "moss").strip().lower()
if TTS_BACKEND not in ("moss", "xtts"):
    logger.warning("Invalid TTS_BACKEND=%r; using 'moss' instead", TTS_BACKEND)
    TTS_BACKEND = "moss"

# MOSS-TTS (the default). Apache-2.0, and its language list covers German and
# Arabic. It speaks with a built-in voice, so no reference clip is needed and
# nothing is re-synthesised per call; a clip can still be supplied via
# MOSS_REF_WAV to clone a specific voice.
MOSS_MODEL = os.getenv("MOSS_MODEL", "OpenMOSS-Team/MOSS-TTS-Local-Transformer").strip()
MOSS_REF_WAV = os.getenv("MOSS_REF_WAV", "").strip()
# RVQ inference depth (max 32). Lower is faster and less detailed - this is the
# speed/quality dial.
MOSS_VQ_DEPTH = _env_int("MOSS_VQ_DEPTH", 32)
MOSS_TEMPERATURE = _env_positive_float("MOSS_TEMPERATURE", 1.0)
MOSS_TOP_P = _env_positive_float("MOSS_TOP_P", 0.95)
MOSS_TOP_K = _env_int("MOSS_TOP_K", 50)
MOSS_REPETITION_PENALTY = _env_positive_float("MOSS_REPETITION_PENALTY", 1.1)
# The model card states the audio codec runs at 12.5 tokens per second.
MOSS_TOKENS_PER_SECOND = 12.5
# A T4 has no bf16 tensor cores, so the pipeline picks fp16 there. MOSS-TTS is
# published with a bf16 example; override this if fp16 ever misbehaves.
MOSS_DTYPE = os.getenv("MOSS_DTYPE", "auto").strip().lower()
XTTS_MODEL_NAME = "tts_models/multilingual/multi-dataset/xtts_v2"
# When set, the TTS engine runs in its own process with this directory first on
# its PYTHONPATH. MOSS-TTS's remote code breaks on transformers >= 5.4 (configs
# became strict dataclasses), while Cohere ASR only exists from 5.4 on, so the
# Kaggle notebook installs transformers 5.3.0 there for the TTS alone.
TTS_WORKER_PYTHONPATH = os.getenv("TTS_WORKER_PYTHONPATH", "").strip()

# Speak each finished clause as soon as the translator has produced it instead
# of waiting for the full sentence. This is the single biggest win for
# time-to-first-audio; disable it if chunk boundaries hurt the prosody.
STREAM_TTS = _env_flag("STREAM_TTS", True)
# Minimum characters before the first / any later clause is handed to the TTS.
# The first chunk is deliberately short so audio starts early; later chunks are
# longer because the TTS sounds better with more context.
TTS_FIRST_CHUNK_CHARS = _env_int("TTS_FIRST_CHUNK_CHARS", 40)
TTS_CHUNK_CHARS = _env_int("TTS_CHUNK_CHARS", 110)
# Force a chunk break once the buffer gets this long, even without punctuation.
TTS_MAX_CHUNK_CHARS = _env_int("TTS_MAX_CHUNK_CHARS", 240)
# Never hand the TTS a fragment shorter than this: the TTS needs a few words of
# context and clips very short inputs.
TTS_MIN_CHUNK_CHARS = _env_int("TTS_MIN_CHUNK_CHARS", 25)

# --- Audio output ----------------------------------------------------------
AUDIO_SAMPLE_RATE = 24_000
AUDIO_FRAME_SAMPLES = AUDIO_SAMPLE_RATE // 50
# LiveKit buffer in front of the encoder. Small enough that our own backlog
# accounting stays accurate, large enough to survive a scheduling hiccup.
AUDIO_QUEUE_MS = _env_int("AUDIO_QUEUE_MS", 800)
# Silence inserted between two turns so they do not run into each other.
TURN_GAP_SECONDS = 0.12
# If the agent falls behind by more than this many seconds of unplayed audio,
# the oldest pending chunks are dropped. Without this the lag grows forever as
# soon as synthesis is slower than real time.
MAX_AUDIO_BACKLOG_SECONDS = _env_positive_float("MAX_AUDIO_BACKLOG_SECONDS", 10.0)

# --- Scheduling ------------------------------------------------------------
# How many model calls may occupy the GPU at once. >1 lets the Python-side
# pre/post-processing of one stage overlap the CUDA kernels of another, which
# is where most of the pipelining win comes from on a single GPU.
GPU_CONCURRENCY = _env_int("GPU_CONCURRENCY", 2)
# Live Arabic captions while someone is still speaking. They are pure extra
# GPU work on the critical path's budget, so they only ever run when the
# ASR -> translation -> TTS path is completely idle.
INTERIM_CAPTIONS = _env_flag("INTERIM_CAPTIONS", True)

PRIO_SPEECH = 0  # TTS: closest to the listener's ear, an underrun is audible
PRIO_TURN = 1  # final ASR and translation
PRIO_INTERIM = 8  # best-effort live captions
PRIO_CRITICAL_MAX = PRIO_TURN

# --- Segmentation ----------------------------------------------------------
SAMPLE_RATE = 16_000
VAD_CHUNK = 512  # Silero VAD requires exactly 32 ms windows at 16 kHz
CHUNK_SECONDS = VAD_CHUNK / SAMPLE_RATE
SPEECH_START_PROB = 0.5
SPEECH_KEEP_PROB = 0.35
# Trailing silence that ends an utterance once it already contains a full
# clause (env END_SILENCE_SECONDS, default 0.4 s).
END_SILENCE_SECONDS = _env_positive_float("END_SILENCE_SECONDS", 0.4)
END_SILENCE_CHUNKS = round(END_SILENCE_SECONDS / CHUNK_SECONDS)
# A speaker pausing for effect mid-sentence must not end the utterance: below
# MIN_SENTENCE_SECONDS of actual speech, a much longer pause is required before
# the segment is closed. Fragmenting sentences is expensive twice over - the
# translator loses the context it needs, and every extra turn pays the TTS
# engine's fixed per-call cost, which is what pushes synthesis past real time.
MIN_SENTENCE_SECONDS = _env_positive_float("MIN_SENTENCE_SECONDS", 2.5)
SHORT_SEGMENT_SILENCE_SECONDS = _env_positive_float("SHORT_SEGMENT_SILENCE_SECONDS", 1.0)
SHORT_SEGMENT_SILENCE_CHUNKS = round(SHORT_SEGMENT_SILENCE_SECONDS / CHUNK_SECONDS)
PRE_ROLL_CHUNKS = round(0.25 / CHUNK_SECONDS)
TAIL_CHUNKS = round(0.15 / CHUNK_SECONDS)
MIN_SPEECH_CHUNKS = round(0.25 / CHUNK_SECONDS)
# Env MAX_UTTERANCE_SECONDS: longest utterance (seconds) sent to the ASR.
MAX_UTTERANCE_SECONDS = _env_positive_float("MAX_UTTERANCE_SECONDS", 12.0)
MAX_UTTERANCE_CHUNKS = round(MAX_UTTERANCE_SECONDS / CHUNK_SECONDS)
SPLIT_SEARCH_CHUNKS = round(2.0 / CHUNK_SECONDS)
INTERIM_STEP_CHUNKS = round(0.6 / CHUNK_SECONDS)

_utterance_counter = itertools.count()
_boot_id = secrets.token_hex(3)


def new_utterance_id() -> str:
    # Unique across bot restarts so the browser never overwrites old lines.
    return f"{_boot_id}-{next(_utterance_counter)}"


# ---------------------------------------------------------------------------
# Speech segmentation
# ---------------------------------------------------------------------------


class Segmenter:
    """Splits a continuous audio stream into utterances using Silero VAD.

    Runs on a worker thread, never on the event loop. Not thread-safe: every
    speaker owns one instance plus one thread.
    """

    def __init__(self) -> None:
        self.vad = load_silero_vad()
        self.pending = np.empty(0, dtype=np.float32)
        self.pre_roll: collections.deque[np.ndarray] = collections.deque(maxlen=PRE_ROLL_CHUNKS)
        self.chunks: list[np.ndarray] = []
        self.probs: list[float] = []
        self.speaking = False
        self.silent_chunks = 0
        self.speech_chunks = 0
        # Cached concatenation of self.chunks so repeated interim snapshots do
        # not re-copy the whole utterance every time (that was quadratic).
        self._snapshot: np.ndarray | None = None

    @property
    def size(self) -> int:
        return len(self.chunks)

    def snapshot(self) -> np.ndarray:
        if self._snapshot is None or self._snapshot.size != self.size * VAD_CHUNK:
            self._snapshot = np.concatenate(self.chunks) if self.chunks else np.empty(0, dtype=np.float32)
        return self._snapshot

    def _append(self, chunk: np.ndarray, prob: float) -> None:
        self.chunks.append(chunk)
        self.probs.append(prob)
        if prob >= SPEECH_KEEP_PROB:
            self.speech_chunks += 1
        self._snapshot = None

    def _silence_needed(self) -> int:
        """Trailing silence (in VAD chunks) required to end the current segment.

        A segment that already holds a full clause ends on a short pause, which
        keeps latency low. A short segment needs a much longer pause, because a
        mid-sentence breath must not be mistaken for the end of a sentence.
        """
        if self.speech_chunks * CHUNK_SECONDS >= MIN_SENTENCE_SECONDS:
            return END_SILENCE_CHUNKS
        return max(END_SILENCE_CHUNKS, SHORT_SEGMENT_SILENCE_CHUNKS)

    @torch.inference_mode()
    def feed(self, pcm: np.ndarray) -> list[np.ndarray | None]:
        """Adds audio and returns finished utterances (None = too short to be speech)."""
        self.pending = np.concatenate((self.pending, pcm)) if self.pending.size else pcm
        finished: list[np.ndarray | None] = []

        offset = 0
        while self.pending.size - offset >= VAD_CHUNK:
            chunk = self.pending[offset:offset + VAD_CHUNK].copy()
            offset += VAD_CHUNK
            prob = float(self.vad(torch.from_numpy(chunk), SAMPLE_RATE))

            if not self.speaking:
                if prob < SPEECH_START_PROB:
                    self.pre_roll.append(chunk)
                    continue
                self.speaking = True
                self.chunks = list(self.pre_roll)
                self.probs = [0.0] * len(self.chunks)
                self.speech_chunks = 0
                self._snapshot = None
                self.pre_roll.clear()
                self.silent_chunks = 0

            self._append(chunk, prob)
            self.silent_chunks = 0 if prob >= SPEECH_KEEP_PROB else self.silent_chunks + 1

            if self.silent_chunks >= self._silence_needed():
                end = self.size - self.silent_chunks + TAIL_CHUNKS
                self.pre_roll.extend(self.chunks[end:])
                finished.append(self._cut(end))
                self.speaking = False
            elif self.size >= MAX_UTTERANCE_CHUNKS:
                # Split long speech at the quietest moment of the last seconds, not mid-word.
                window = self.probs[-SPLIT_SEARCH_CHUNKS:]
                end = self.size - len(window) + int(np.argmin(window)) + 1
                finished.append(self._cut(end))
                self.silent_chunks = 0
        self.pending = self.pending[offset:]
        return finished

    def flush(self) -> np.ndarray | None:
        if not self.speaking:
            return None
        self.speaking = False
        return self._cut(self.size)

    def _cut(self, end: int) -> np.ndarray | None:
        """Returns the first `end` chunks as an utterance and keeps the rest."""
        head_chunks, head_probs = self.chunks[:end], self.probs[:end]
        self.chunks, self.probs = self.chunks[end:], self.probs[end:]
        self.speech_chunks = sum(p >= SPEECH_KEEP_PROB for p in self.probs)
        self._snapshot = None
        if sum(p >= SPEECH_KEEP_PROB for p in head_probs) < MIN_SPEECH_CHUNKS:
            return None
        return np.concatenate(head_chunks)


class SpeakerStream:
    """One participant: a VAD thread plus the state the pipeline needs.

    The LiveKit audio callback only drops raw PCM into a queue; all VAD work
    happens on a dedicated thread so the event loop stays free to run the
    pipeline stages. The thread drains every frame that arrived in the meantime
    in one batch, which keeps up even if the loop was blocked for a moment.
    """

    _STOP = object()

    def __init__(self, identity: str, loop: asyncio.AbstractEventLoop, on_utterance) -> None:
        self.identity = identity
        self.utterance_id = new_utterance_id()
        self._loop = loop
        self._on_utterance = on_utterance
        self._inbox: queue.SimpleQueue = queue.SimpleQueue()
        self._lock = threading.Lock()
        self._segmenter: Segmenter | None = None
        # Published by the VAD thread, read by the interim worker.
        self.speaking = False
        self.pending_chunks = 0
        self.pending_speech_chunks = 0
        self._thread = threading.Thread(target=self._run, name=f"vad-{identity}", daemon=True)
        self._thread.start()

    def push(self, pcm: np.ndarray) -> None:
        self._inbox.put(pcm)

    def close(self) -> None:
        self._inbox.put(self._STOP)

    def snapshot(self) -> np.ndarray | None:
        """Thread-safe copy of the audio received for the current utterance."""
        with self._lock:
            if self._segmenter is None or not self._segmenter.speaking:
                return None
            return self._segmenter.snapshot()

    def next_utterance(self) -> None:
        self.utterance_id = new_utterance_id()

    def _run(self) -> None:
        segmenter = Segmenter()
        with self._lock:
            self._segmenter = segmenter
        try:
            while True:
                item = self._inbox.get()
                if item is self._STOP:
                    break
                batch = [item]
                stopped = False
                while True:
                    try:
                        nxt = self._inbox.get_nowait()
                    except queue.Empty:
                        break
                    if nxt is self._STOP:
                        stopped = True
                        break
                    batch.append(nxt)
                pcm = batch[0] if len(batch) == 1 else np.concatenate(batch)
                with self._lock:
                    finished = segmenter.feed(pcm)
                    self.speaking = segmenter.speaking
                    self.pending_chunks = segmenter.size
                    self.pending_speech_chunks = segmenter.speech_chunks
                for audio in finished:
                    self._emit(audio)
                if stopped:
                    break
            with self._lock:
                tail = segmenter.flush()
                self.speaking = False
            if tail is not None:
                self._emit(tail)
        except Exception:  # noqa: BLE001 - a dead VAD thread must be visible
            logger.exception("VAD thread for %s crashed", self.identity)

    def _emit(self, audio: np.ndarray | None) -> None:
        self._loop.call_soon_threadsafe(self._on_utterance, self, audio, time.monotonic())


# ---------------------------------------------------------------------------
# GPU scheduling
# ---------------------------------------------------------------------------


class GpuScheduler:
    """Priority admission control for the single shared GPU.

    A bounded number of model calls may run at once: CUDA interleaves their
    kernels and the Python-side pre/post-processing of one stage overlaps the
    kernels of another, while the bound keeps activation memory from spiking
    into an OOM. Waiters are served by priority, so a best-effort live caption
    can never delay the ASR -> translation -> TTS path of a finished utterance.
    """

    def __init__(self, concurrency: int) -> None:
        self._slots = max(1, concurrency)
        self._active = 0
        self._waiting: list[tuple[int, int, asyncio.Future]] = []
        self._seq = itertools.count()
        self.critical_inflight = 0

    @property
    def busy(self) -> bool:
        return self.critical_inflight > 0 or bool(self._waiting)

    @contextlib.asynccontextmanager
    async def reserve(self, priority: int):
        await self._acquire(priority)
        critical = priority <= PRIO_CRITICAL_MAX
        if critical:
            self.critical_inflight += 1
        try:
            yield
        finally:
            if critical:
                self.critical_inflight -= 1
            self._release()

    async def run(self, priority: int, func, *args):
        """Runs a blocking model call on a worker thread under a GPU slot."""
        async with self.reserve(priority):
            return await asyncio.to_thread(func, *args)

    async def _acquire(self, priority: int) -> None:
        if self._active < self._slots and not self._waiting:
            self._active += 1
            return
        future = asyncio.get_running_loop().create_future()
        heapq.heappush(self._waiting, (priority, next(self._seq), future))
        try:
            await future
        except asyncio.CancelledError:
            if future.done() and not future.cancelled():
                # The slot was handed over right before the cancellation
                # arrived; give it back instead of leaking it.
                self._release()
            raise

    def _release(self) -> None:
        while self._waiting:
            _, _, future = heapq.heappop(self._waiting)
            if not future.done():
                future.set_result(None)  # hands the slot over, _active unchanged
                return
        self._active -= 1


# ---------------------------------------------------------------------------
# Model loading helpers
# ---------------------------------------------------------------------------


def pick_device_dtype(requested: str = "auto") -> tuple[torch.dtype, str]:
    if requested == "cpu" or (requested == "auto" and not torch.cuda.is_available()):
        logger.warning("Using CPU for this stage; model inference will be slow.")
        return torch.float32, "cpu"
    device = "cuda:0" if requested in ("auto", "cuda") else requested
    if not re.fullmatch(r"cuda:\d+", device):
        raise ValueError(f"Invalid device {requested!r}; use auto, cpu, or cuda:N.")
    index = int(device.split(":")[1])
    if not torch.cuda.is_available() or index >= torch.cuda.device_count():
        raise RuntimeError(f"Requested {device}, but this runtime does not expose that GPU.")
    major, _ = torch.cuda.get_device_capability(index)
    # A T4 is sm_75 and has no bf16 tensor cores, so fp16 is the fast path there.
    return (torch.bfloat16 if major >= 8 else torch.float16), f"cuda:{index}"


def _dtype_kwargs(dtype) -> dict:
    """`dtype=` on transformers >= 4.56, `torch_dtype=` on older releases."""
    import transformers

    parts = transformers.__version__.split(".")
    major = int(parts[0])
    minor = int(parts[1]) if len(parts) > 1 else 0
    if major > 4 or (major == 4 and minor >= 56):
        return {"dtype": dtype}
    return {"torch_dtype": dtype}


def load_with_sdpa(loader, name: str, **kwargs):
    """Loads a model with SDPA attention, retrying without it if unsupported."""
    try:
        return loader.from_pretrained(name, attn_implementation="sdpa", **kwargs)
    except (ValueError, ImportError) as exc:
        if "attn_implementation" not in str(exc) and "scaled_dot_product" not in str(exc):
            raise
        logger.info("%s has no SDPA kernel; falling back to the eager attention.", name)
        return loader.from_pretrained(name, **kwargs)


def _four_bit_config(dtype):
    from transformers import BitsAndBytesConfig

    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=dtype,
    )


def _vram_report(stage: str, device: str = "cuda:0") -> None:
    if not torch.cuda.is_available() or device == "cpu":
        return
    index = int(device.split(":")[1])
    free, total = torch.cuda.mem_get_info(index)
    logger.info("VRAM after %s on %s: %.1f GiB used of %.1f GiB", stage, device,
                (total - free) / 2**30, total / 2**30)


def _is_oom(exc: BaseException) -> bool:
    return isinstance(exc, torch.cuda.OutOfMemoryError) or "out of memory" in str(exc).lower()


def _device_map(device: str):
    """Explicit single-device placement.

    device_map="auto" silently offloads layers to the CPU when VRAM looks
    tight and then runs an order of magnitude slower; an explicit map either
    fits on the GPU or raises an OOM we can react to.
    """
    return {"": int(device.split(":")[1])} if device != "cpu" else {"": "cpu"}


def _check_moss_memory(dtype, device: str) -> None:
    """Reject configurations whose known weight floor exceeds the entire GPU."""
    if TTS_BACKEND != "moss":
        return
    if MOSS_MODEL != "OpenMOSS-Team/MOSS-TTS-Local-Transformer":
        return
    # Published safetensors metadata, not the "1.7B" architecture label.
    # The processor loads a separate float32 MOSS-Audio-Tokenizer.
    tts_dtype, tts_device = pick_device_dtype(TTS_DEVICE)
    if tts_device == "cpu":
        return
    bytes_per_weight = 4 if tts_dtype == torch.float32 else 2
    moss_bytes = {"auto": bytes_per_weight, "float16": 2, "bfloat16": 2, "float32": 4}
    if MOSS_DTYPE not in moss_bytes:
        raise ValueError("MOSS_DTYPE must be auto, float16, bfloat16, or float32.")
    minimum = 3_060_606_464 * moss_bytes[MOSS_DTYPE] + 1_774_566_400 * 4
    if device == tts_device and COHERE_MODEL_NAME == "CohereLabs/cohere-transcribe-arabic-07-2026":
        minimum += 2_065_804_048 * (0.5 if COHERE_4BIT else (4 if dtype == torch.float32 else 2))
    _, translation_device = pick_device_dtype(TRANSLATION_DEVICE)
    if translation_device == tts_device and TRANSLATION_MODEL == "Qwen/Qwen3-4B-Instruct-2507":
        # Even the automatic 4-bit fallback needs at least this much.
        minimum += 4_022_468_096 * 0.5
    if translation_device == tts_device and TRANSLATION_MODEL == "google/translategemma-12b-it":
        minimum += 12_000_000_000 * 0.5
    total = torch.cuda.get_device_properties(int(tts_device.split(":")[1])).total_memory
    logger.info(
        "Known model weight floor on %s: %.1f GiB; GPU total: %.1f GiB. "
        "Quantization metadata, caches and inference need additional memory.",
        tts_device, minimum / 2**30, total / 2**30,
    )
    if minimum >= total:
        raise RuntimeError(
            f"MOSS + codec + configured ASR/translator require at least {minimum / 2**30:.1f} GiB "
            f"of weights, but this GPU has {total / 2**30:.1f} GiB total. "
            "This configuration cannot fit a T4. Select a larger GPU "
            "(32 GiB or more recommended; Colab A100 40 GB), then run all cells. "
            "No models have been loaded."
        )


def _bf16_works(device: str) -> bool:
    """Runs a real bfloat16 matmul; GPUs without bf16 tensor cores may emulate it."""
    try:
        probe = torch.ones((16, 16), dtype=torch.bfloat16, device=device)
        return bool(torch.isfinite(probe @ probe).all().item())
    except Exception:  # noqa: BLE001 - any failure means "not usable"
        logger.debug("bfloat16 probe failed on %s", device, exc_info=True)
        return False


def translation_dtype(default, device: str):
    """Translator activation dtype; see TRANSLATION_DTYPE."""
    if TRANSLATION_DTYPE != "auto":
        if TRANSLATION_DTYPE not in ("float16", "bfloat16", "float32"):
            raise ValueError("TRANSLATION_DTYPE must be auto, float16, bfloat16, or float32.")
        return getattr(torch, TRANSLATION_DTYPE)
    if TRANSLATION_BACKEND != "translategemma" or device == "cpu" or default != torch.float16:
        return default
    # Unsloth documents that Gemma 3 (1B-27B) activations exceed float16's range
    # on T4-class GPUs. bfloat16 has float32's range; without bf16 tensor cores
    # it is slower but correct.
    if _bf16_works(device):
        logger.info("TranslateGemma: using bfloat16 instead of float16 (Gemma 3 overflows float16).")
        return torch.bfloat16
    logger.warning("bfloat16 is not usable on %s; TranslateGemma falls back to float32 "
                   "(more VRAM, slower).", device)
    return torch.float32


def stage_schedulers() -> tuple[GpuScheduler, GpuScheduler, GpuScheduler]:
    """Share admission slots only between stages on the same device."""
    devices = [pick_device_dtype(value)[1] for value in
               (ASR_DEVICE, TRANSLATION_DEVICE, TTS_DEVICE)]
    schedulers = {device: GpuScheduler(GPU_CONCURRENCY) for device in set(devices)}
    return schedulers[devices[0]], schedulers[devices[1]], schedulers[devices[2]]


# ---------------------------------------------------------------------------
# ASR
# ---------------------------------------------------------------------------


class AsrEngine:
    """Cohere Transcribe Arabic. Blocking calls, serialised by its own lock."""

    topic = TRANSCRIPT_TOPIC

    def __init__(self, dtype, device: str) -> None:
        from transformers import AutoProcessor, CohereAsrForConditionalGeneration

        load_kwargs: dict = {"device_map": _device_map(device)}
        if HF_TOKEN:
            load_kwargs["token"] = HF_TOKEN
        if COHERE_4BIT:
            load_kwargs["quantization_config"] = _four_bit_config(dtype)
        else:
            load_kwargs.update(_dtype_kwargs(dtype))
        try:
            self.processor = AutoProcessor.from_pretrained(COHERE_MODEL_NAME, token=HF_TOKEN)
            self.model = load_with_sdpa(
                CohereAsrForConditionalGeneration, COHERE_MODEL_NAME, **load_kwargs
            )
        except Exception as exc:  # noqa: BLE001 - wrapped with actionable guidance below
            raise RuntimeError(
                "Could not load the Cohere model: "
                f"{exc}. The repo is gated: accept the terms on huggingface.co and "
                "set HF_TOKEN in the .env."
            ) from exc
        self.model.eval()
        self.model.generation_config.use_cache = True
        self._lock = threading.Lock()

    def transcribe(self, audio: np.ndarray) -> str:
        seconds = audio.size / SAMPLE_RATE
        # Budget generously: speech tops out around five words per second and
        # Arabic script costs a few tokens per word, so this only ever stops a
        # hallucinating decode - it must never clip a normal transcript.
        budget = min(MAX_GEN_TOKENS, int(seconds * ASR_TOKENS_PER_SECOND) + 48)
        with self._lock:
            inputs = self.processor(
                audio, sampling_rate=SAMPLE_RATE, return_tensors="pt", language=COHERE_LANGUAGE
            )
            inputs = inputs.to(self.model.device, dtype=self.model.dtype)
            with torch.inference_mode():
                start = time.perf_counter()
                outputs = self.model.generate(**inputs, max_new_tokens=budget, do_sample=False)
            ms = (time.perf_counter() - start) * 1000
            decoded = self.processor.decode(outputs, skip_special_tokens=True)
        if isinstance(decoded, list):
            decoded = decoded[0] if decoded else ""
        # Encoder-decoder ASR models have no input_ids (audio features only) and
        # generate() returns decoder tokens only; decoder-only models prepend the
        # prompt, so subtract its length when present.
        input_ids = inputs.get("input_ids")
        prompt_len = input_ids.shape[1] if input_ids is not None else 0
        generated = outputs.shape[1] - prompt_len
        if generated >= budget:
            logger.warning(
                "ASR hit its %d-token budget for %.1f s of audio - the transcript is cut off. "
                "Raise MAX_GEN_TOKENS or ASR_TOKENS_PER_SECOND.",
                budget, seconds,
            )
        logger.debug("ASR %.0f ms for %.1f s audio (%d tokens)", ms, seconds, generated)
        return decoded.strip()


# ---------------------------------------------------------------------------
# Translation
# ---------------------------------------------------------------------------


class Translator:
    """Qwen3 or TranslateGemma translator with token streaming.

    Streaming matters more than raw tokens/second here: the pipeline can start
    speaking the first clause while the rest is still being generated.
    """

    def __init__(self, dtype, device: str) -> None:
        from transformers import AutoProcessor, AutoTokenizer

        self.backend = TRANSLATION_BACKEND
        load_kwargs: dict = {}
        if HF_TOKEN:
            load_kwargs["token"] = HF_TOKEN
        if self.backend == "translategemma":
            self.processor = AutoProcessor.from_pretrained(TRANSLATION_MODEL, **load_kwargs)
            self.tokenizer = self.processor.tokenizer
        else:
            self.tokenizer = AutoTokenizer.from_pretrained(TRANSLATION_MODEL, **load_kwargs)
        self.quantized = TRANSLATION_4BIT
        self.dtype = dtype
        # Set by stream(): True when the last request ran into its token budget.
        self.last_hit_budget = False
        if TRANSLATION_4BIT:
            self.model = self._load(dtype, device, load_kwargs, four_bit=True)
        else:
            try:
                self.model = self._load(dtype, device, load_kwargs, four_bit=False)
            except Exception as exc:  # noqa: BLE001 - only OOM is recoverable
                if not _is_oom(exc):
                    raise
                logger.warning(
                    "%s does not fit next to the ASR and TTS models in fp16/bf16 (%s); falling back to "
                    "4-bit. 4-bit decodes slower and translates slightly worse - use a GPU with "
                    "more VRAM or set COHERE_4BIT=1 to free headroom.",
                    TRANSLATION_MODEL,
                    exc.__class__.__name__,
                )
                self.quantized = True
            if self.quantized:
                # Leave the exception scope before retrying: its traceback can
                # retain tensors from the failed full-precision load.
                torch.cuda.empty_cache()
                self.model = self._load(dtype, device, load_kwargs, four_bit=True)
        self.model.eval()
        self.model.generation_config.use_cache = True
        self._lock = threading.Lock()

    @staticmethod
    def _load(dtype, device: str, load_kwargs: dict, four_bit: bool):
        from transformers import AutoModelForCausalLM, AutoModelForImageTextToText

        extra = _dtype_kwargs(dtype)
        if four_bit:
            extra["quantization_config"] = _four_bit_config(dtype)
        return load_with_sdpa(
            AutoModelForImageTextToText if TRANSLATION_BACKEND == "translategemma" else AutoModelForCausalLM,
            TRANSLATION_MODEL,
            device_map=_device_map(device),
            **extra,
            **load_kwargs,
        )

    def _build_inputs(self, text: str, context: list[tuple[str, str]]):
        if self.backend == "translategemma":
            # TranslateGemma's official template accepts structured translation
            # requests, not a system prompt or arbitrary chat instructions.
            messages = [{
                "role": "user",
                "content": [{
                    "type": "text",
                    "source_lang_code": TRANSLATION_SOURCE_LANG,
                    "target_lang_code": TRANSLATION_TARGET_LANG,
                    "text": text,
                }],
            }]
            inputs = self.processor.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True,
                return_dict=True, return_tensors="pt",
            )
            if inputs["input_ids"].shape[1] > 2048:
                raise ValueError(
                    "TranslateGemma input exceeds its documented 2048-token input context. "
                    "Shorten MAX_UTTERANCE_SECONDS; source text was not truncated."
                )
            return inputs.to(self.model.device)
        messages: list[dict[str, str]] = []
        if TRANSLATION_SYSTEM_PROMPT:
            messages.append({"role": "system", "content": TRANSLATION_SYSTEM_PROMPT})
        # Interleaved source/translation pairs. Sending all sources first and
        # all translations afterwards (as an earlier version did) destroys the
        # few-shot pattern and measurably degrades the output.
        for previous_source, previous_target in context:
            messages.append({"role": "user", "content": previous_source})
            messages.append({"role": "assistant", "content": previous_target})
        messages.append({"role": "user", "content": text})
        # enable_thinking=False: some Qwen3 instruct models think with a
        # reasoning block by default, which is slow and can eat the whole output
        # budget before reaching the translation. Templates without a thinking
        # branch simply ignore the kwarg.
        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=False,
            return_dict=True,
            return_tensors="pt",
        ).to(self.model.device)

    def stream(self, text: str, context: list[tuple[str, str]], emit) -> None:
        """Generates the translation and calls `emit(piece)` for each new piece."""
        from transformers import TextIteratorStreamer

        with self._lock:
            inputs = self._build_inputs(text, context)
            input_len = inputs["input_ids"].shape[1]
            source_tokens = len(self.tokenizer(text)["input_ids"])
            # German can be noticeably longer than the Arabic source, so budget
            # generously: this cap exists to stop a degenerate repetition loop,
            # not to shorten a legitimate translation.
            budget = max(
                64, min(MAX_GEN_TOKENS, int(source_tokens * TRANSLATION_TOKEN_RATIO) + 48)
            )
            streamer = TextIteratorStreamer(
                self.tokenizer, skip_prompt=True, skip_special_tokens=True, timeout=120.0
            )
            error: list[BaseException] = []
            produced: list[int] = []

            def generate() -> None:
                try:
                    with torch.inference_mode():
                        output = self.model.generate(
                            **inputs,
                            max_new_tokens=budget,
                            do_sample=False,
                            repetition_penalty=1.05,
                            streamer=streamer,
                        )
                    produced.append(output.shape[1] - input_len)
                except BaseException as exc:  # noqa: BLE001 - re-raised on the caller's thread
                    error.append(exc)
                    streamer.end()

            worker = threading.Thread(target=generate, name="translator-generate", daemon=True)
            worker.start()
            try:
                for piece in streamer:
                    if piece:
                        emit(piece)
            finally:
                worker.join()
            if error:
                raise error[0]
            self.last_hit_budget = bool(produced and produced[0] >= budget)
            if self.last_hit_budget:
                logger.warning(
                    "Translation hit its %d-token budget for a %d-token source - the output is "
                    "cut off. Raise TRANSLATION_TOKEN_RATIO or MAX_GEN_TOKENS.",
                    budget, source_tokens,
                )

    def translate(self, text: str, context: list[tuple[str, str]] = ()) -> str:
        parts: list[str] = []
        self.stream(text, list(context), parts.append)
        return "".join(parts).strip()


# Strong clause boundaries (sentence end, colon, semicolon, newline) and the
# weak ones used only when a chunk would otherwise grow too long.
_STRONG_BREAK = re.compile(r"[.!?؟…:;\n]")
_WEAK_BREAK = re.compile(r"[,،)\u00bb\"]")


def speakable_target(index: int) -> int:
    """Target length of the chunk with this index within a translation.

    The first chunk is deliberately short so audio starts early; later chunks
    grow quickly because every TTS call carries a fixed per-call cost, so
    splitting one sentence into many chunks multiplies the GPU work.
    """
    if index <= 0:
        return TTS_FIRST_CHUNK_CHARS
    return min(TTS_MAX_CHUNK_CHARS, TTS_CHUNK_CHARS * index)


def split_speakable(buffer: str, index: int, flush: bool) -> tuple[str, str]:
    """Splits off the next chunk that is worth speaking.

    Returns (chunk, rest); an empty chunk means "keep buffering". The cut is
    the *earliest* clause boundary, so audio starts as soon as a speakable unit
    exists instead of waiting for the longest one that still fits. The target
    length only decides when to start looking: a complete sentence that falls
    somewhat short of it is still a better chunk than a padded one.
    """
    text = buffer.lstrip()
    if not text:
        return "", ""
    if flush:
        return text, ""
    target = speakable_target(index)
    if len(text) < target:
        return "", buffer
    # The TTS needs a few words of context to sound natural, so never cut below
    # this even when a sentence ends very early.
    floor = max(TTS_MIN_CHUNK_CHARS, target // 2)

    window = text[: max(target, TTS_MAX_CHUNK_CHARS)]
    cut = min((m.end() for m in _STRONG_BREAK.finditer(window) if m.end() >= floor), default=0)
    if not cut and len(text) >= TTS_MAX_CHUNK_CHARS:
        cut = min((m.end() for m in _WEAK_BREAK.finditer(window) if m.end() >= floor), default=0)
        if not cut:
            cut = window.rfind(" ")
    if cut <= 0:
        return "", buffer
    return text[:cut].strip(), text[cut:]


# ---------------------------------------------------------------------------
# TTS
# ---------------------------------------------------------------------------


def _resample(audio: np.ndarray, source_rate: int, target_rate: int) -> np.ndarray:
    if source_rate == target_rate or audio.size == 0:
        return audio
    duration = audio.size / source_rate
    target_len = max(1, int(round(duration * target_rate)))
    source_t = np.linspace(0.0, duration, num=audio.size, endpoint=False, dtype=np.float64)
    target_t = np.linspace(0.0, duration, num=target_len, endpoint=False, dtype=np.float64)
    return np.interp(target_t, source_t, audio).astype(np.float32)


class XttsSynthesizer:
    """XTTS-v2 voice (only used when TTS_BACKEND=xtts)."""

    def __init__(self) -> None:
        self.model = None
        self.speaker = os.getenv("XTTS_SPEAKER", "").strip() or None
        self.speaker_wav = os.getenv("XTTS_SPEAKER_WAV", "").strip() or None
        self.sample_rate = AUDIO_SAMPLE_RATE

    def load(self):
        if self.model is not None:
            return self.model
        if self.speaker_wav and not Path(self.speaker_wav).is_file():
            raise RuntimeError(f"XTTS speaker file not found: {self.speaker_wav}")
        from TTS.api import TTS

        _, device = pick_device_dtype(TTS_DEVICE)
        self.model = TTS(XTTS_MODEL_NAME, gpu=False, progress_bar=False).to(device)
        if not self.speaker and not self.speaker_wav:
            speakers = self.model.speakers or []
            if not speakers:
                raise RuntimeError("XTTS has no default voice; configure XTTS_SPEAKER_WAV.")
            self.speaker = speakers[0]
        rate = getattr(getattr(self.model, "synthesizer", None), "output_sample_rate", None)
        self.sample_rate = int(rate or AUDIO_SAMPLE_RATE)
        return self.model

    def synthesize(self, text: str) -> np.ndarray:
        model = self.load()
        options = {"text": text, "language": TRANSLATION_TARGET_LANG}
        if self.speaker_wav:
            options["speaker_wav"] = self.speaker_wav
        elif self.speaker:
            options["speaker"] = self.speaker
        audio = np.asarray(model.tts(**options), dtype=np.float32).reshape(-1)
        return _resample(audio, self.sample_rate, AUDIO_SAMPLE_RATE)


class MossSynthesizer:
    """MOSS-TTS (Apache-2.0).

    Its language list covers German and Arabic, and it has a built-in voice:
    no reference clip is required and none is re-synthesised per call. Setting
    MOSS_REF_WAV switches it to zero-shot cloning of that voice.
    """

    def __init__(self) -> None:
        self.model = None
        self.processor = None
        self.sample_rate = AUDIO_SAMPLE_RATE
        self._reference: list[str] | None = None

    def load(self):
        if self.model is not None:
            return self.model
        from transformers import AutoModel, AutoProcessor

        dtype, device = pick_device_dtype(TTS_DEVICE)
        if MOSS_DTYPE != "auto":
            if MOSS_DTYPE not in ("float16", "bfloat16", "float32"):
                raise ValueError("MOSS_DTYPE must be auto, float16, bfloat16, or float32.")
            dtype = getattr(torch, MOSS_DTYPE)
        if dtype == torch.bfloat16 and device != "cpu" and torch.cuda.get_device_capability(int(device.split(":")[1]))[0] < 8:
            raise ValueError("MOSS_DTYPE=bfloat16 is not supported on a T4; use auto or float32.")
        if not 1 <= MOSS_VQ_DEPTH <= 32:
            raise ValueError("MOSS_VQ_DEPTH must be between 1 and 32.")
        if MOSS_TOP_P > 1:
            raise ValueError("MOSS_TOP_P must be greater than 0 and at most 1.")
        if MOSS_REF_WAV and not Path(MOSS_REF_WAV).is_file():
            raise RuntimeError(f"MOSS reference file not found: {MOSS_REF_WAV}")
        if device != "cpu":
            # The model card calls the cuDNN SDPA backend broken and disables
            # it, keeping the other kernels as fallbacks.
            torch.backends.cuda.enable_cudnn_sdp(False)
        attn = "sdpa" if device != "cpu" else "eager"
        logger.info("Loading MOSS-TTS %s (%s, %s) ...", MOSS_MODEL, dtype, attn)
        try:
            # No extra kwargs: MOSS's processor forwards every kwarg into its own
            # constructor, which rejects unknown ones such as `token`. The MOSS
            # repos are public; downloads still pick up HF_TOKEN from the env.
            processor = AutoProcessor.from_pretrained(MOSS_MODEL, trust_remote_code=True)
            # Keep the codec's published float32 dtype; its quantizer explicitly
            # casts activations to float32. Include it in VRAM planning.
            processor.audio_tokenizer = processor.audio_tokenizer.to(device).eval()
            model = AutoModel.from_pretrained(
                MOSS_MODEL,
                trust_remote_code=True,
                token=HF_TOKEN,
                device_map=_device_map(device),
                attn_implementation=attn,
                **_dtype_kwargs(dtype),
            )
            model.eval()
            # transformers 5.3.0 (unlike 5.0.0, which MOSS targets) pre-builds
            # DynamicCache(config=get_text_config()) for every generate(); MOSS's
            # top-level config keeps num_hidden_layers inside language_config, so
            # that raises AttributeError. "dynamic_full" builds a config-free
            # DynamicCache, as 5.0.0 effectively did. MOSS copies this config on
            # every generate() call.
            model.generation_config.cache_implementation = "dynamic_full"
        except Exception as exc:  # noqa: BLE001 - wrapped with actionable guidance
            raise RuntimeError(
                f"Could not load MOSS-TTS ({MOSS_MODEL}) and its audio codec: "
                f"{type(exc).__name__}: {exc} (full traceback above)."
            ) from exc
        self.processor = processor
        self.model = model
        self.sample_rate = int(processor.model_config.sampling_rate)
        if MOSS_REF_WAV:
            self._reference = [MOSS_REF_WAV]
            logger.info("MOSS-TTS clones the voice from %s", MOSS_REF_WAV)
        else:
            logger.info("MOSS-TTS uses its built-in voice (no reference clip needed)")
        return self.model

    def _token_budget(self, text: str) -> int:
        """Audio tokens to allow for this text, with a wide safety margin.

        German is spoken at roughly 14 characters per second; the margin is
        deliberately large because a budget that bites would clip the end of a
        sentence, and a finished utterance stops on its own long before it.
        """
        seconds = len(text) / 14.0
        return max(96, int(seconds * MOSS_TOKENS_PER_SECOND * 2.5) + 64)

    def synthesize(self, text: str) -> np.ndarray:
        self.load()
        message = (
            self.processor.build_user_message(text=text, reference=self._reference)
            if self._reference
            else self.processor.build_user_message(text=text)
        )
        batch = self.processor([[message]], mode="generation")
        device = self.model.device
        budget = self._token_budget(text)
        kwargs = {
            "input_ids": batch["input_ids"].to(device),
            "attention_mask": batch["attention_mask"].to(device),
            "max_new_tokens": budget,
        }
        tuning = {
            "n_vq_for_inference": MOSS_VQ_DEPTH,
            "audio_temperature": MOSS_TEMPERATURE,
            "audio_top_p": MOSS_TOP_P,
            "audio_top_k": MOSS_TOP_K,
            "audio_repetition_penalty": MOSS_REPETITION_PENALTY,
        }
        with torch.inference_mode():
            outputs = self.model.generate(**kwargs, **tuning)

            audio = None
            for decoded in self.processor.decode(outputs):
                codes = getattr(decoded, "audio_codes_list", None)
                if codes:
                    audio = codes[0]
                    break
        if audio is None:
            raise RuntimeError("MOSS-TTS returned no audio for a non-empty text.")
        waveform = np.asarray(audio.detach().to(torch.float32).cpu().numpy()).reshape(-1)
        if waveform.size == 0 or not np.isfinite(waveform).all():
            raise RuntimeError(
                "MOSS-TTS returned empty or non-finite audio. If using fp16, "
                "try MOSS_DTYPE=float32 (requires more GPU memory)."
            )
        produced_tokens = waveform.size / self.sample_rate * MOSS_TOKENS_PER_SECOND
        if produced_tokens >= budget * 0.98:
            logger.warning(
                "MOSS-TTS hit its %d-token budget - the audio is probably cut off. "
                "Report this; the budget is derived from the text length.", budget,
            )
        return _resample(waveform, self.sample_rate, AUDIO_SAMPLE_RATE)


class TtsWorkerClient:
    """Runs the TTS engine in a child process (TTS_WORKER_PYTHONPATH).

    The child gets its own transformers version and its own GIL, so MOSS
    decoding on one GPU never competes with the translator's decode loop for
    the interpreter. Protocol: one JSON line per request on stdin; one JSON
    header line, then raw float32 samples, per response on stdout.
    """

    def __init__(self) -> None:
        self.process: subprocess.Popen | None = None
        self.sample_rate = AUDIO_SAMPLE_RATE
        self.command = [sys.executable, "-u", str(Path(__file__).resolve()), "--tts-worker"]

    def load(self):
        if self.process is not None:
            return self.process
        env = os.environ.copy()
        env["PYTHONPATH"] = os.pathsep.join(
            path for path in (TTS_WORKER_PYTHONPATH, env.get("PYTHONPATH", "")) if path
        )
        logger.info("Starting the %s TTS worker process (PYTHONPATH %s) ...",
                    TTS_BACKEND, TTS_WORKER_PYTHONPATH)
        # stderr is inherited, so the worker's logs and tracebacks appear inline.
        self.process = subprocess.Popen(
            self.command, stdin=subprocess.PIPE, stdout=subprocess.PIPE, env=env,
        )
        atexit.register(self.close)
        header = self._read_header()
        if not header.get("ok"):
            raise RuntimeError(f"TTS worker could not load the model: {header.get('error')}")
        logger.info("TTS worker ready (pid %d, transformers %s)",
                    self.process.pid, header.get("transformers"))
        return self.process

    def synthesize(self, text: str) -> np.ndarray:
        process = self.load()
        try:
            process.stdin.write(json.dumps({"text": text}).encode("utf-8") + b"\n")
            process.stdin.flush()
        except (BrokenPipeError, OSError) as exc:
            raise self._died() from exc
        header = self._read_header()
        if not header.get("ok"):
            raise RuntimeError(f"TTS worker failed: {header.get('error')}")
        size = int(header["bytes"])
        data = process.stdout.read(size)
        if len(data) != size:
            raise self._died()
        return np.frombuffer(data, dtype=np.float32).copy()

    def _read_header(self) -> dict:
        line = self.process.stdout.readline()
        if not line:
            raise self._died()
        return json.loads(line)

    def _died(self) -> RuntimeError:
        code = None
        if self.process is not None:
            # EOF can arrive a moment before the exit status is available.
            with contextlib.suppress(subprocess.TimeoutExpired):
                code = self.process.wait(timeout=5)
        return RuntimeError(
            f"TTS worker process exited (code {code}); its traceback is in the log above."
        )

    def close(self) -> None:
        """Ends the worker and waits, so its GPU memory is free afterwards."""
        process, self.process = self.process, None
        if process is None:
            return
        with contextlib.suppress(OSError):
            process.stdin.close()
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=10)


def run_tts_worker() -> int:
    """Child-process side of TtsWorkerClient."""
    # Keep the real stdout for the protocol and send everything else that
    # libraries print to stderr, so no stray output can corrupt a response.
    protocol = os.fdopen(os.dup(1), "wb")
    os.dup2(2, 1)

    def send(header: dict, payload: bytes = b"") -> None:
        protocol.write(json.dumps(header).encode("utf-8") + b"\n" + payload)
        protocol.flush()

    try:
        import transformers

        engine = SpeechSynthesizer._ENGINES[TTS_BACKEND]()
        engine.load()
    except Exception as exc:  # noqa: BLE001 - reported to the parent
        logger.exception("TTS worker failed to load")
        send({"ok": False, "error": f"{type(exc).__name__}: {exc}"})
        return 1
    send({"ok": True, "transformers": transformers.__version__})
    for line in sys.stdin.buffer:
        try:
            text = json.loads(line)["text"]
            audio = np.ascontiguousarray(engine.synthesize(text), dtype=np.float32)
        except Exception as exc:  # noqa: BLE001 - one failed chunk must not end the worker
            logger.exception("TTS worker request failed")
            send({"ok": False, "error": f"{type(exc).__name__}: {exc}"})
            continue
        data = audio.tobytes()
        send({"ok": True, "bytes": len(data)}, data)
    return 0


class SpeechSynthesizer:
    """Spoken output for the translation; the engine is picked by TTS_BACKEND."""

    _ENGINES = {"moss": MossSynthesizer, "xtts": XttsSynthesizer}

    def __init__(self) -> None:
        if TTS_WORKER_PYTHONPATH:
            self.engine = TtsWorkerClient()
        else:
            self.engine = self._ENGINES.get(TTS_BACKEND, MossSynthesizer)()
        self._lock = threading.Lock()

    def load(self) -> None:
        with self._lock:
            self.engine.load()

    def synthesize(self, text: str) -> np.ndarray:
        with self._lock:
            start = time.perf_counter()
            audio = self.engine.synthesize(text)
            ms = (time.perf_counter() - start) * 1000
        seconds = audio.size / AUDIO_SAMPLE_RATE
        logger.debug("TTS %.0f ms for %.1f s audio (rtf %.2f)", ms, seconds, (ms / 1000) / max(seconds, 1e-6))
        return audio


# ---------------------------------------------------------------------------
# Pipeline
# ---------------------------------------------------------------------------


@dataclass
class Turn:
    """One utterance travelling through the pipeline."""

    identity: str
    utterance_id: str
    audio: np.ndarray | None
    captured_at: float
    source_text: str = ""
    translation: str = ""
    wants_audio: bool = False
    asr_ms: float = 0.0
    translation_ms: float = 0.0
    first_token_ms: float = 0.0
    tts_ms: float = 0.0
    audio_seconds: float = 0.0
    first_audio_at: float | None = None
    chunks: int = 0


@dataclass
class AudioChunk:
    turn: Turn
    samples: np.ndarray
    last: bool = False
    tail_gap: float = field(default=TURN_GAP_SECONDS)


class Pipeline:
    """Stage workers for one LiveKit room.

    Every stage owns a task and a queue, so an utterance can be transcribed
    while the previous one is being translated, spoken, and played back. The
    playout stage is deliberately separate: it runs in real time and must never
    be able to stall a GPU stage, which is what made the old single-loop design
    drift further behind with every turn.
    """

    def __init__(self, asr: AsrEngine, translator: Translator, speech: SpeechSynthesizer,
                 room: rtc.Room, audio_source: rtc.AudioSource, scheduler: GpuScheduler,
                 translation_scheduler: GpuScheduler | None = None,
                 tts_scheduler: GpuScheduler | None = None) -> None:
        self.asr = asr
        self.translator = translator
        self.speech = speech
        self.room = room
        self.audio_source = audio_source
        self.gpu = scheduler
        self.translation_gpu = translation_scheduler if translation_scheduler is not None else scheduler
        self.tts_gpu = tts_scheduler if tts_scheduler is not None else scheduler

        self.asr_queue: asyncio.Queue[Turn] = asyncio.Queue()
        self.translation_queue: asyncio.Queue[Turn] = asyncio.Queue()
        self.tts_queue: asyncio.Queue[tuple[Turn, str, bool]] = asyncio.Queue()
        self.playout_queue: asyncio.Queue[AudioChunk] = asyncio.Queue()

        self.speakers: dict[str, SpeakerStream] = {}
        self.audio_enabled: dict[str, bool] = {}
        self.translation_context: collections.deque[tuple[str, str]] = collections.deque(
            maxlen=TRANSLATION_CONTEXT_TURNS or 1
        )
        self._interim_sizes: dict[str, int] = {}
        self._playout_backlog = 0.0
        self._playing_turn: str | None = None
        self._background: set[asyncio.Task] = set()

    def _spawn(self, coro) -> asyncio.Task:
        """Fire-and-forget send that must not hold up a GPU slot."""
        task = asyncio.create_task(coro)
        self._background.add(task)
        task.add_done_callback(self._background.discard)
        return task

    # -- stage 0: VAD callback ---------------------------------------------

    def on_utterance(self, speaker: SpeakerStream, audio: np.ndarray | None, captured_at: float) -> None:
        turn = Turn(speaker.identity, speaker.utterance_id, audio, captured_at)
        speaker.next_utterance()
        self._interim_sizes.pop(speaker.identity, None)
        self.asr_queue.put_nowait(turn)

    # -- stage 1: ASR -------------------------------------------------------

    async def run_asr(self) -> None:
        while True:
            turn = await self.asr_queue.get()
            try:
                if turn.audio is None:
                    await self._send(turn.identity, "final", turn.utterance_id, "", TRANSCRIPT_TOPIC)
                    continue
                started = time.monotonic()
                try:
                    text = await self.gpu.run(PRIO_TURN, self.asr.transcribe, turn.audio)
                except Exception:
                    logger.exception("Transcription failed")
                    continue
                turn.asr_ms = (time.monotonic() - started) * 1000
                turn.source_text = (text or "").strip()
                await self._send(turn.identity, "final", turn.utterance_id, turn.source_text, TRANSCRIPT_TOPIC)
                if turn.source_text:
                    turn.wants_audio = self.audio_enabled.get(turn.identity, False)
                    self.translation_queue.put_nowait(turn)
            finally:
                self.asr_queue.task_done()

    # -- stage 1b: best-effort live captions --------------------------------

    async def run_interim(self) -> None:
        """Live Arabic captions, only ever while the critical path is idle.

        Re-transcribing a growing buffer is quadratic work, so this stage polls
        at a low rate and yields the GPU to anything on the ASR -> translation
        -> TTS path.
        """
        if not INTERIM_CAPTIONS:
            return
        while True:
            speaker = self._next_interim()
            if speaker is None or self._critical_busy():
                await asyncio.sleep(0.15)
                continue
            audio = speaker.snapshot()
            if audio is None or audio.size == 0:
                await asyncio.sleep(0.15)
                continue
            utterance_id = speaker.utterance_id
            self._interim_sizes[speaker.identity] = speaker.pending_chunks
            try:
                text = await self.gpu.run(PRIO_INTERIM, self.asr.transcribe, audio)
            except Exception:
                logger.exception("Interim transcription failed")
                await asyncio.sleep(0.5)
                continue
            # Skip if the utterance was finalised meanwhile; its final follows.
            if text and utterance_id == speaker.utterance_id:
                await self._send(speaker.identity, "interim", utterance_id, text, TRANSCRIPT_TOPIC)

    def _critical_busy(self) -> bool:
        return (
            self.gpu.busy
            or self.translation_gpu.busy
            or self.tts_gpu.busy
            or self.asr_queue.qsize() > 0
            or self.translation_queue.qsize() > 0
            or self.tts_queue.qsize() > 0
        )

    def _next_interim(self) -> SpeakerStream | None:
        candidates = [
            s for s in self.speakers.values()
            if s.speaking
            and s.pending_chunks - self._interim_sizes.get(s.identity, 0) >= INTERIM_STEP_CHUNKS
            and s.pending_speech_chunks >= MIN_SPEECH_CHUNKS
        ]
        return max(
            candidates,
            key=lambda s: s.pending_chunks - self._interim_sizes.get(s.identity, 0),
            default=None,
        )

    # -- stage 2: translation ----------------------------------------------

    async def run_translation(self) -> None:
        while True:
            turn = await self.translation_queue.get()
            try:
                await self._translate(turn)
            except Exception:
                logger.exception("Translation for %s failed", turn.identity)
            finally:
                self.translation_queue.task_done()

    async def _translate(self, turn: Turn) -> None:
        loop = asyncio.get_running_loop()
        pieces: asyncio.Queue[str | None] = asyncio.Queue()
        context = list(self.translation_context) if TRANSLATION_CONTEXT_TURNS and TRANSLATION_BACKEND == "qwen" else []
        started = time.monotonic()
        failure: list[BaseException] = []

        def emit(piece: str) -> None:
            loop.call_soon_threadsafe(pieces.put_nowait, piece)

        def generate() -> None:
            try:
                self.translator.stream(turn.source_text, context, emit)
            except BaseException as exc:  # noqa: BLE001 - surfaced below
                failure.append(exc)
            finally:
                loop.call_soon_threadsafe(pieces.put_nowait, None)

        async with self.translation_gpu.reserve(PRIO_TURN):
            worker = asyncio.create_task(asyncio.to_thread(generate))
            buffer = ""
            spoken = ""
            chunk_index = 0
            last_published = 0.0
            partial: asyncio.Task | None = None
            try:
                while True:
                    piece = await pieces.get()
                    if piece is None:
                        break
                    if not turn.first_token_ms:
                        turn.first_token_ms = (time.monotonic() - started) * 1000
                    turn.translation += piece
                    buffer += piece
                    # Live translation text: the browser replaces the line by
                    # id. Sent fire-and-forget so the network never holds the
                    # GPU slot this stage is sitting on.
                    now = time.monotonic()
                    if now - last_published > 0.15:
                        last_published = now
                        partial = self._spawn(self._send(
                            turn.identity, "translation", turn.utterance_id,
                            turn.translation.strip(), TRANSLATION_TOPIC,
                        ))
                    if turn.wants_audio and STREAM_TTS:
                        while True:
                            chunk, buffer = split_speakable(buffer, chunk_index, flush=False)
                            if not chunk:
                                break
                            chunk_index += 1
                            spoken += chunk
                            self.tts_queue.put_nowait((turn, chunk, False))
            finally:
                await worker
        if failure:
            raise failure[0]

        turn.translation = turn.translation.strip()
        turn.translation_ms = (time.monotonic() - started) * 1000
        if partial is not None:
            # The last partial must be on the wire before the final text, or it
            # could overwrite the complete translation in the browser.
            await asyncio.gather(partial, return_exceptions=True)
        await self._send(
            turn.identity, "translation", turn.utterance_id, turn.translation, TRANSLATION_TOPIC
        )
        if turn.translation and TRANSLATION_CONTEXT_TURNS and TRANSLATION_BACKEND == "qwen":
            self.translation_context.append((turn.source_text, turn.translation))

        if not turn.wants_audio or not turn.translation:
            self._log_turn(turn)
            return
        rest = buffer.strip() if STREAM_TTS else turn.translation
        if rest:
            self.tts_queue.put_nowait((turn, rest, True))
        elif STREAM_TTS and spoken:
            self.tts_queue.put_nowait((turn, "", True))
        else:
            self._log_turn(turn)

    # -- stage 3: TTS -------------------------------------------------------

    async def run_tts(self) -> None:
        while True:
            turn, text, last = await self.tts_queue.get()
            samples = np.zeros(0, dtype=np.float32)
            try:
                if text:
                    started = time.monotonic()
                    samples = await self.tts_gpu.run(PRIO_SPEECH, self.speech.synthesize, text)
                    turn.tts_ms += (time.monotonic() - started) * 1000
                    turn.chunks += 1
                    turn.audio_seconds += samples.size / AUDIO_SAMPLE_RATE
            except Exception:
                logger.exception("Speech synthesis for %s failed", turn.identity)
                samples = np.zeros(0, dtype=np.float32)
            finally:
                self.tts_queue.task_done()
            # Exactly one chunk per turn carries `last`, and only the playout
            # stage reports the turn, so the timing line is logged once.
            if samples.size or last:
                self._enqueue_playout(AudioChunk(turn, samples, last))

    # -- stage 4: playout ---------------------------------------------------

    def _enqueue_playout(self, chunk: AudioChunk) -> None:
        self._playout_backlog += chunk.samples.size / AUDIO_SAMPLE_RATE
        self.playout_queue.put_nowait(chunk)
        if self._playout_backlog > MAX_AUDIO_BACKLOG_SECONDS:
            self._drop_backlog()

    def _drop_backlog(self) -> None:
        """Drops whole queued turns so the agent can catch up.

        Only reachable when synthesis is persistently slower than the incoming
        speech; without it the spoken output drifts behind forever. Dropping
        happens per turn, never per chunk: losing one complete sentence is
        recoverable, a sentence cut in half mid-word is not. The turn currently
        being spoken and the newest turn are always kept.
        """
        target = MAX_AUDIO_BACKLOG_SECONDS / 2
        pending: list[AudioChunk] = []
        while not self.playout_queue.empty():
            pending.append(self.playout_queue.get_nowait())
            self.playout_queue.task_done()

        groups: list[tuple[Turn, list[AudioChunk]]] = []
        for chunk in pending:
            if groups and groups[-1][0] is chunk.turn:
                groups[-1][1].append(chunk)
            else:
                groups.append((chunk.turn, [chunk]))

        dropped_seconds = 0.0
        dropped_turns: list[str] = []
        kept: list[AudioChunk] = []
        for index, (turn, chunks) in enumerate(groups):
            droppable = (
                index < len(groups) - 1
                and turn.utterance_id != self._playing_turn
                and self._playout_backlog - dropped_seconds > target
            )
            if droppable:
                dropped_seconds += sum(c.samples.size for c in chunks) / AUDIO_SAMPLE_RATE
                dropped_turns.append(turn.utterance_id)
                continue
            kept.extend(chunks)
        for chunk in kept:
            self.playout_queue.put_nowait(chunk)

        if dropped_turns:
            logger.warning(
                "Speech output was %.1f s behind; skipped %d whole turn(s) (%.1f s): %s. "
                "Lower MOSS_VQ_DEPTH or shorten MAX_UTTERANCE_SECONDS if this repeats.",
                self._playout_backlog, len(dropped_turns), dropped_seconds, ", ".join(dropped_turns),
            )
            self._playout_backlog -= dropped_seconds
        elif self._playout_backlog > MAX_AUDIO_BACKLOG_SECONDS:
            logger.warning(
                "Speech output is %.1f s behind but nothing can be skipped without cutting a "
                "sentence in half; the lag will persist until the speaker pauses.",
                self._playout_backlog,
            )

    async def run_playout(self) -> None:
        """Pushes synthesised audio to LiveKit in real time.

        capture_frame() blocks at the speed of playback, which is exactly why
        this must be its own task and must not hold a GPU slot.
        """
        while True:
            chunk = await self.playout_queue.get()
            try:
                self._playing_turn = chunk.turn.utterance_id
                if chunk.samples.size:
                    if chunk.turn.first_audio_at is None:
                        chunk.turn.first_audio_at = time.monotonic()
                    await self._stream_samples(chunk.samples)
                    self._playout_backlog = max(
                        0.0, self._playout_backlog - chunk.samples.size / AUDIO_SAMPLE_RATE
                    )
                if chunk.last:
                    if chunk.tail_gap > 0:
                        await self._stream_samples(
                            np.zeros(int(chunk.tail_gap * AUDIO_SAMPLE_RATE), dtype=np.float32)
                        )
                    self._playing_turn = None
                    self._log_turn(chunk.turn)
            except Exception:
                logger.exception("Audio playout failed")
            finally:
                self.playout_queue.task_done()

    async def _stream_samples(self, waveform: np.ndarray) -> None:
        samples = (np.clip(np.nan_to_num(waveform), -1.0, 1.0) * 32767).astype(np.int16)
        for offset in range(0, samples.size, AUDIO_FRAME_SAMPLES):
            frame_samples = np.ascontiguousarray(samples[offset:offset + AUDIO_FRAME_SAMPLES])
            if frame_samples.size == 0:
                continue
            frame = rtc.AudioFrame(
                data=frame_samples.tobytes(),
                sample_rate=AUDIO_SAMPLE_RATE,
                num_channels=1,
                samples_per_channel=frame_samples.size,
            )
            await self.audio_source.capture_frame(frame)

    # -- reporting ----------------------------------------------------------

    def _log_turn(self, turn: Turn) -> None:
        speech = (turn.audio.size / SAMPLE_RATE) if turn.audio is not None else 0.0
        parts = [
            f"turn {turn.utterance_id}",
            f"speech {speech:.1f}s",
            f"asr {turn.asr_ms:.0f}ms",
            f"mt {turn.translation_ms:.0f}ms (first token {turn.first_token_ms:.0f}ms)",
        ]
        if turn.chunks:
            rtf = (turn.tts_ms / 1000) / max(turn.audio_seconds, 1e-6)
            parts.append(
                f"tts {turn.tts_ms:.0f}ms in {turn.chunks} chunk(s) "
                f"-> {turn.audio_seconds:.1f}s audio (rtf {rtf:.2f})"
            )
        if turn.first_audio_at is not None:
            parts.append(f"first audio {turn.first_audio_at - turn.captured_at:.2f}s after speech end")
        else:
            parts.append(f"text ready {time.monotonic() - turn.captured_at:.2f}s after speech end")
        parts.append(f"backlog {self._playout_backlog:.1f}s")
        logger.info(" | ".join(parts))

    # -- LiveKit plumbing ---------------------------------------------------

    async def _send(self, identity: str, kind: str, utterance_id: str, text: str, topic: str) -> None:
        try:
            await self.room.local_participant.publish_data(
                json.dumps({"type": kind, "id": utterance_id, "text": text}, ensure_ascii=False),
                reliable=True,
                destination_identities=[identity],
                topic=topic,
            )
        except Exception as exc:  # noqa: BLE001 - a dropped caption must not kill the stage
            logger.warning("Sending to %s failed: %s", identity, exc)

    def on_data(self, packet: rtc.DataPacket) -> None:
        if packet.topic != SETTINGS_TOPIC or packet.participant is None:
            return
        try:
            message = json.loads(packet.data.decode("utf-8"))
        except (UnicodeDecodeError, json.JSONDecodeError):
            return
        if message.get("type") != "settings":
            return
        enabled = message.get("audio_output")
        if isinstance(enabled, bool):
            self.audio_enabled[packet.participant.identity] = enabled

    async def listen(self, track: rtc.Track, identity: str) -> None:
        loop = asyncio.get_running_loop()
        speaker = SpeakerStream(identity, loop, self.on_utterance)
        self.speakers[track.sid] = speaker
        self._interim_sizes.pop(identity, None)
        stream = rtc.AudioStream(track, sample_rate=SAMPLE_RATE, num_channels=1)
        logger.info("Transcribing %s", identity)
        try:
            async for event in stream:
                # The event loop only converts and forwards; the VAD itself runs
                # on the speaker's own thread.
                pcm = np.frombuffer(event.frame.data, dtype=np.int16).astype(np.float32) / 32768.0
                speaker.push(pcm)
        finally:
            self.speakers.pop(track.sid, None)
            self._interim_sizes.pop(identity, None)
            # Non-blocking: the VAD thread flushes its tail utterance and exits
            # on its own, so this stays safe inside a cancelled task.
            speaker.close()
            with contextlib.suppress(Exception):
                await stream.aclose()
            logger.info("%s finished", identity)

    def workers(self) -> list:
        return [
            asyncio.create_task(self.run_asr(), name="asr"),
            asyncio.create_task(self.run_interim(), name="interim"),
            asyncio.create_task(self.run_translation(), name="translate"),
            asyncio.create_task(self.run_tts(), name="tts"),
            asyncio.create_task(self.run_playout(), name="playout"),
        ]


# ---------------------------------------------------------------------------
# Startup
# ---------------------------------------------------------------------------


def run_smoke_test(asr: AsrEngine, translator: Translator, speech: SpeechSynthesizer,
                   output_dir: str, audio_path: str | None = None) -> None:
    """Save real model outputs and sequential timings, without a LiveKit room."""
    import soundfile as sf

    destination = Path(output_dir)
    destination.mkdir(parents=True, exist_ok=True)
    source = "الحمد لله رب العالمين. نسأل الله أن يرزقنا العلم النافع والعمل الصالح."
    asr_seconds = None
    input_seconds = None
    if audio_path:
        audio, rate = sf.read(audio_path, dtype="float32", always_2d=True)
        audio = _resample(audio.mean(axis=1), rate, SAMPLE_RATE)
        input_seconds = audio.size / SAMPLE_RATE
        if not 0 < input_seconds <= MAX_UTTERANCE_SECONDS or not np.isfinite(audio).all():
            raise ValueError(
                f"Smoke-test audio must be finite and between 0 and {MAX_UTTERANCE_SECONDS} seconds."
            )
        started = time.monotonic()
        source = asr.transcribe(audio)
        asr_seconds = time.monotonic() - started
        if not source.strip():
            raise RuntimeError("Smoke-test ASR returned no text for the supplied audio.")
    started = time.monotonic()
    first_token_seconds = None
    pieces: list[str] = []

    def emit(piece: str) -> None:
        nonlocal first_token_seconds
        if piece and first_token_seconds is None:
            first_token_seconds = time.monotonic() - started
        pieces.append(piece)

    translator.stream(source, [], emit)
    translation_seconds = time.monotonic() - started
    translated = "".join(pieces).strip()
    if not translated:
        raise RuntimeError("Smoke-test translation returned no text.")
    buffer = translated
    chunks: list[np.ndarray] = []
    chunk_times: list[float] = []
    while buffer:
        chunk, rest = split_speakable(buffer, len(chunks), flush=False)
        if not chunk:
            chunk, rest = split_speakable(buffer, len(chunks), flush=True)
        started = time.monotonic()
        samples = speech.synthesize(chunk)
        if samples.size == 0 or not np.isfinite(samples).all():
            raise RuntimeError("Smoke-test TTS returned empty or non-finite audio.")
        chunk_times.append(time.monotonic() - started)
        chunks.append(samples)
        buffer = rest
    waveform = np.concatenate(chunks)
    audio_seconds = waveform.size / AUDIO_SAMPLE_RATE
    sf.write(destination / "german.wav", waveform, AUDIO_SAMPLE_RATE, subtype="PCM_16")
    metrics = {
        "translation_model": TRANSLATION_MODEL,
        "translation_backend": TRANSLATION_BACKEND,
        "translation_4bit": translator.quantized,
        "translation_dtype": str(getattr(translator, "dtype", None)),
        "tts_model": MOSS_MODEL if TTS_BACKEND == "moss" else XTTS_MODEL_NAME,
        "source_text": source,
        "german_text": translated,
        "asr_tested_on_supplied_audio": bool(audio_path),
        "input_audio_seconds": input_seconds,
        "asr_seconds": asr_seconds,
        "translation_seconds": translation_seconds,
        "first_token_seconds": first_token_seconds,
        "tts_seconds": sum(chunk_times),
        "tts_chunk_seconds": chunk_times,
        "output_audio_seconds": audio_seconds,
        "tts_rtf": sum(chunk_times) / audio_seconds,
        "timing_scope": "Sequential offline test after warm-up; NOT live end-to-end latency.",
    }
    if torch.cuda.is_available():
        metrics["gpu_peak_allocated_gib"] = {
            f"cuda:{index}": torch.cuda.max_memory_allocated(index) / 2**30
            for index in range(torch.cuda.device_count())
        }
    (destination / "metrics.json").write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2) + "\n", encoding="utf-8",
    )
    logger.info("Smoke test saved to %s | MT %.2fs | TTS RTF %.2f (must be <1 to keep up).",
                destination, translation_seconds, metrics["tts_rtf"])
    if not audio_path:
        logger.warning("Smoke test used Arabic text, not microphone audio; ASR accuracy was NOT tested.")
    if metrics["tts_rtf"] >= 1:
        logger.warning("TTS is slower than real time on this sample. Live STS will build a backlog.")


def prepare_models() -> tuple[AsrEngine, Translator, SpeechSynthesizer]:
    dtype, device = pick_device_dtype(ASR_DEVICE)
    default_translation_dtype, translation_device = pick_device_dtype(TRANSLATION_DEVICE)
    _, tts_device = pick_device_dtype(TTS_DEVICE)
    _check_moss_memory(dtype, device)
    logger.info("Placement: ASR=%s | translation=%s | TTS/codec=%s",
                device, translation_device, tts_device)
    if TRANSLATION_BACKEND == "translategemma":
        logger.info(
            "TranslateGemma uses its official %s->%s template; "
            "Qwen system prompt and rolling context are not used.",
            TRANSLATION_SOURCE_LANG, TRANSLATION_TARGET_LANG,
        )
    rng = np.random.default_rng(0)
    warmup_audio = (rng.standard_normal(SAMPLE_RATE) * 0.01).astype(np.float32)

    logger.info("Loading Cohere ASR %s%s", COHERE_MODEL_NAME, " (4-bit)" if COHERE_4BIT else f" ({dtype})")
    asr = AsrEngine(dtype, device)
    asr.transcribe(warmup_audio)
    _vram_report("ASR", device)

    load_silero_vad()  # Cache the VAD weights before the first speaker thread starts.

    # Reserve TTS and codec memory before the translator decides whether full
    # precision fits. Otherwise TTS is the last load and has no OOM fallback.
    speech = SpeechSynthesizer()
    logger.info("Preloading %s so the first spoken turn is fast ...", TTS_BACKEND)
    speech.load()
    audio = speech.synthesize("Guten Morgen.")
    if audio.size == 0 or not np.isfinite(audio).all():
        raise RuntimeError("TTS warm-up produced empty or non-finite audio; startup aborted.")
    _vram_report("TTS", tts_device)

    compute_dtype = translation_dtype(default_translation_dtype, translation_device)
    logger.info("Loading translator %s (%s, %s compute)", TRANSLATION_MODEL,
                "4-bit" if TRANSLATION_4BIT else "full precision", compute_dtype)
    translator = Translator(compute_dtype, translation_device)
    if not translator.translate("صباح الخير.").strip():
        if getattr(translator, "last_hit_budget", False):
            raise RuntimeError(
                "Translator warm-up generated tokens up to its budget, but all were special "
                f"tokens (computed in {getattr(translator, 'dtype', '?')}). This is the typical "
                "sign of NaN logits from numeric overflow; set TRANSLATION_DTYPE=float32."
            )
        raise RuntimeError("Translator warm-up produced an empty translation; startup aborted.")
    logger.info("Translator ready: %s, %s compute", "4-bit" if translator.quantized else "full precision",
                getattr(translator, "dtype", "?"))
    _vram_report("translator", translation_device)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return asr, translator, speech


def ws_url(url: str) -> str:
    if url.startswith("https://"):
        return "wss://" + url[len("https://"):]
    if url.startswith("http://"):
        return "ws://" + url[len("http://"):]
    return url


def bot_token(key: str, secret: str) -> str:
    return (
        api.AccessToken(key, secret)
        .with_identity(BOT_IDENTITY)
        .with_name("Arabic ASR Bot")
        .with_grants(
            api.VideoGrants(
                room_join=True,
                room=ROOM_NAME,
                can_publish=True,
                can_subscribe=True,
                can_publish_data=True,
            )
        )
        .to_jwt()
    )


async def check_credentials(url: str, key: str, secret: str) -> None:
    lkapi = api.LiveKitAPI(url=url, api_key=key, api_secret=secret)
    try:
        await lkapi.room.list_rooms(api.ListRoomsRequest())
    finally:
        await lkapi.aclose()


async def run_room(asr: AsrEngine, translator: Translator, speech: SpeechSynthesizer,
                   url: str, key: str, secret: str) -> None:
    room = rtc.Room()
    audio_source = rtc.AudioSource(
        sample_rate=AUDIO_SAMPLE_RATE, num_channels=1, queue_size_ms=AUDIO_QUEUE_MS
    )
    scheduler, translation_scheduler, tts_scheduler = stage_schedulers()
    pipeline = Pipeline(asr, translator, speech, room, audio_source, scheduler,
                        translation_scheduler, tts_scheduler)
    stage_workers = pipeline.workers()
    listeners: dict[str, asyncio.Task] = {}
    disconnected = asyncio.Event()

    def start_listening(track: rtc.Track, participant: rtc.RemoteParticipant) -> None:
        if track.kind != rtc.TrackKind.KIND_AUDIO or track.sid in listeners:
            return
        task = asyncio.create_task(pipeline.listen(track, participant.identity))
        listeners[track.sid] = task
        task.add_done_callback(lambda _: listeners.pop(track.sid, None))

    def stop_listening(track: rtc.Track) -> None:
        task = listeners.get(track.sid)
        if task:
            task.cancel()

    room.on("track_subscribed", lambda track, _pub, participant: start_listening(track, participant))
    room.on("track_unsubscribed", lambda track, _pub, _participant: stop_listening(track))
    room.on("data_received", pipeline.on_data)
    room.on("disconnected", lambda *_: disconnected.set())

    try:
        await room.connect(ws_url(url), bot_token(key, secret))
        audio_track = rtc.LocalAudioTrack.create_audio_track("german-translation", audio_source)
        await room.local_participant.publish_track(audio_track)
        logger.info(
            "READY: Bot is in room '%s' (streaming pipeline, %d slot(s) per device). "
            "Now open the web app and start the microphone.",
            ROOM_NAME,
            GPU_CONCURRENCY,
        )
        for participant in room.remote_participants.values():
            for publication in participant.track_publications.values():
                if publication.track is not None:
                    start_listening(publication.track, participant)
        await disconnected.wait()
    finally:
        tasks = [*listeners.values(), *stage_workers, *pipeline._background]
        for task in tasks:
            task.cancel()
        await asyncio.gather(*tasks, return_exceptions=True)
        await room.disconnect()
        await audio_source.aclose()


async def main(smoke_dir: str | None = None, test_audio: str | None = None) -> None:
    # asyncio.to_thread shares one executor; Colab's 2 vCPUs would otherwise
    # cap it at six threads for ASR, translation, TTS and the VAD handoff.
    asyncio.get_running_loop().set_default_executor(
        ThreadPoolExecutor(max_workers=8, thread_name_prefix="model")
    )
    url = os.environ.get("LIVEKIT_URL", "").strip()
    key = os.environ.get("LIVEKIT_API_KEY", "").strip()
    secret = os.environ.get("LIVEKIT_API_SECRET", "").strip()
    missing = [n for n, v in (("LIVEKIT_URL", url), ("LIVEKIT_API_KEY", key), ("LIVEKIT_API_SECRET", secret)) if not v]
    if missing and smoke_dir is None:
        raise SystemExit(f"Missing in .env: {', '.join(missing)}")

    if smoke_dir is None:
        logger.info("Checking LiveKit credentials ...")
        try:
            await check_credentials(url, key, secret)
        except Exception as exc:
            raise SystemExit(f"LiveKit rejected the credentials or is not reachable: {exc}")
        logger.info("LiveKit credentials are valid.")
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(index)
            logger.info("GPU %d: %s (%.1f GiB)", index, props.name, props.total_memory / 2**30)
    logger.info("ASR model: %s", COHERE_MODEL_NAME)
    logger.info("Translation model: %s (%s)", TRANSLATION_MODEL, "4-bit" if TRANSLATION_4BIT else "fp16/bf16")
    if TTS_BACKEND == "moss":
        logger.info(
            "TTS backend: moss (%s, RVQ depth %d, voice: %s)",
            MOSS_MODEL, MOSS_VQ_DEPTH, MOSS_REF_WAV or "built-in",
        )
    else:
        logger.info("TTS backend: xtts")
    logger.info(
        "Pipeline: %d GPU slot(s), streaming TTS %s, live captions %s",
        GPU_CONCURRENCY,
        "on" if STREAM_TTS else "off",
        "on" if INTERIM_CAPTIONS else "off",
    )
    logger.info("Max utterance length: %.2f s", MAX_UTTERANCE_SECONDS)
    logger.info("End-of-speech silence: %.2f s", END_SILENCE_SECONDS)
    logger.info("Max generated tokens per pass: %d", MAX_GEN_TOKENS)
    if not HF_TOKEN:
        logger.warning("Cohere ASR is enabled but HF_TOKEN is missing - the repo is gated, so loading will fail.")

    asr, translator, speech = await asyncio.to_thread(prepare_models)
    if smoke_dir is not None:
        await asyncio.to_thread(run_smoke_test, asr, translator, speech, smoke_dir, test_audio)
        return
    while True:
        try:
            await run_room(asr, translator, speech, url, key, secret)
            logger.warning("Connection ended, retrying in 5 seconds")
        except Exception:
            logger.exception("Connection error, retrying in 5 seconds")
        await asyncio.sleep(5)


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--smoke-test", metavar="OUTPUT_DIR",
                        help="Run offline model checks, save German audio and metrics, then exit.")
    parser.add_argument("--test-audio", metavar="PATH",
                        help="Optional short Arabic WAV/FLAC for the offline ASR check.")
    parser.add_argument("--tts-worker", action="store_true",
                        help="Internal: serve TTS requests over stdin/stdout (see TTS_WORKER_PYTHONPATH).")
    args = parser.parse_args()
    if args.tts_worker:
        sys.exit(run_tts_worker())
    if args.test_audio and args.smoke_test is None:
        parser.error("--test-audio requires --smoke-test OUTPUT_DIR")
    try:
        asyncio.run(main(args.smoke_test, args.test_audio))
    except KeyboardInterrupt:
        pass


In [ ]:
from pathlib import Path

# Guards against a stale /content/bot.py from an earlier run: the streaming
# pipeline is what makes the spoken output keep up, so fail loudly without it.
source = Path("/content/bot.py").read_text(encoding="utf-8")
missing = [name for name in ("class Pipeline", "class GpuScheduler", "def split_speakable")
           if name not in source]
if missing:
    raise RuntimeError(
        f"/content/bot.py is out of date (missing: {', '.join(missing)}) - "
        "re-run the %%writefile cell above so it contains the current bot."
    )
print("OK: bot.py is the streaming pipeline (ASR | translation | TTS | playout).")


In [ ]:
import os, subprocess, sys
from pathlib import Path
from dotenv import dotenv_values

if not os.path.exists("/content/bot.py") or not os.path.exists("/content/.env"):
    raise RuntimeError("bot.py or .env is missing. Please use 'Runtime > Run all'.")

env = os.environ.copy()
# The selected .env wins over stale environment values from an earlier run.
env.update({k: v for k, v in dotenv_values("/content/.env", interpolate=False).items() if v is not None})
env["PYTHONUNBUFFERED"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"
env["TF_CPP_MIN_LOG_LEVEL"] = "3"
# Set before the child process touches CUDA: a fragmented caching allocator is
# the usual reason a T4 runs out of memory mid-session even though the totals fit.
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("Starting Cohere ASR + TTS/codec + Qwen3. Model weights download into this Colab runtime.")
print("READY is emitted only after all model warm-ups succeed.")
print("Each turn is reported as: asr | mt | tts | first audio | backlog.")
log_path = Path("/content/bot.log")
print(f"Full output is also saved to {log_path} (copy/download it before resetting the runtime).")
print("Waiting for the 'READY' line ...\n")
stopped = False
with log_path.open("a", encoding="utf-8", buffering=1) as log:
    log.write("\n--- Bot start ---\n")
    process = subprocess.Popen(
        [sys.executable, "-u", "/content/bot.py"], cwd="/content",
        stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env,
    )
    try:
        for line in process.stdout:
            log.write(line)
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        stopped = True
        print("\nStopping bot ...")
    finally:
        if process.poll() is None:
            process.terminate()
        try:
            code = process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            code = process.wait(timeout=5)
        process.stdout.close()
        log.write(f"\nBot exit code: {code}\n")
if not stopped:
    raise RuntimeError(f"Bot exited unexpectedly (code {code}). See the traceback above and /content/bot.log.")
print("Bot stopped.")